In [1]:
# Existing Kaggle dataset input path (no download or KaggleHub call).
from pathlib import Path
DATASET_ROOT = Path("/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset")
if not DATASET_ROOT.is_dir():
    DATASET_ROOT = Path("/kaggle/input/garbage-classification-image-dataset")
print("Dataset:", DATASET_ROOT)


Dataset: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset


In [2]:
# ==========================================================
# EfficientNet-Lite0 Stage 2: same frozen dataset and sustainability protocol
# No encoded manifest, extra ZIP input, or energy optimization.
# ==========================================================
import os, sys, subprocess, importlib.util, time, random, copy, hashlib, json, zipfile, platform
from pathlib import Path
from collections import defaultdict

# Set GPU visibility before importing torch or CodeCarbon.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for module, package in [("timm", "timm"), ("codecarbon", "codecarbon==3.3.2"),
                        ("docx", "python-docx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from importlib.metadata import version as installed_version
if installed_version("codecarbon") != "3.3.2":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "codecarbon==3.3.2"])
import timm, codecarbon
from codecarbon import EmissionsTracker
from docx import Document

SEEDS = [7, 49, 112, 345, 476]
BATCH_SIZE, MAX_EPOCHS, PATIENCE, MIN_DELTA = 32, 10, 3, 1e-4
CARBON_FACTOR = 0.475   # kg CO2e/kWh; fixed global assumption, not Kaggle region
MONITOR_SECONDS, MIN_INFERENCE_REPEATS = 15, 20
OUT = Path("/kaggle/working/REMNET_Stage2")
(OUT / "CodeCarbon_Logs").mkdir(parents=True, exist_ok=True)
(OUT / "Model_Checkpoints").mkdir(exist_ok=True)

# ----------------------------
# Dataset: same Kaggle path as uploaded notebook
# ----------------------------
# DATASET_ROOT is set to the attached Kaggle folder in the first cell.
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError("Kaggle input folder is unavailable; attach the original dataset in Kaggle Add Input.")
search_roots = [DATASET_ROOT]
for archive in sorted(DATASET_ROOT.rglob("*.zip")):
    target = Path("/kaggle/working/remnet_source") / archive.stem
    if not (target / ".extract_complete").exists():
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                if not (target / entry.filename).resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe ZIP entry: {entry.filename}")
            z.extractall(target)
        (target / ".extract_complete").touch()
    search_roots.append(target)

split_dirs = []
for root in search_roots:
    for parent in [root, *sorted((p for p in root.rglob("*") if p.is_dir()), key=str)]:
        children = {p.name.lower(): p for p in parent.iterdir() if p.is_dir()}
        tr, te = children.get("training") or children.get("train"), children.get("testing") or children.get("test")
        if tr and te and {p.name for p in tr.iterdir() if p.is_dir()} == {p.name for p in te.iterdir() if p.is_dir()}:
            split_dirs.append((tr, te))
if not split_dirs:
    raise FileNotFoundError("Training/Testing class folders were not found in the original Kaggle dataset.")
TRAIN_DIR, TEST_DIR = split_dirs[0]
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
assert len(CLASS_NAMES) == 6, f"Expected 6 classes, got {CLASS_NAMES}"
print("Source Training:", TRAIN_DIR, "\nSource Testing:", TEST_DIR)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Stage 1 cleaning: one representative per identical-image group;
# quarantine groups with conflicting labels, and remove the four
# previously visually confirmed near-duplicates.
# The four exclusions are plain filenames, not an encoded manifest.
visual_exclusions = {
    "Training/metal/metal394.jpg", "Training/paper/paper345.jpg",
    "Training/paper/paper390.jpg", "Training/plastic/plastic137.jpg"
}
by_hash = defaultdict(list)
for partition, root in [("Training", TRAIN_DIR), ("Testing", TEST_DIR)]:
    for cls in CLASS_NAMES:
        for path in sorted((root / cls).iterdir()):
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                by_hash[file_sha256(path)].append({"path": path, "class": cls,
                                      "source": f"{partition}/{cls}/{path.name}",
                                      "original_partition": partition})

cleaned = []
conflicts = 0
for digest, group in by_hash.items():
    if len({item["class"] for item in group}) != 1:
        conflicts += len(group)     # Exclude conflicting-label images entirely.
        continue
    item = min(group, key=lambda x: x["source"])  # Prefer Testing over Training.
    cleaned.append({**item, "sha256": digest})

# The Stage 1 80/20 stratified train/validation split was formed BEFORE
# removing 4 visually confirmed near-duplicates. Sort SHA-256 first to
# reproduce exactly the Stage 1 split without embedding any manifest.
original_test = sorted((x for x in cleaned if x["original_partition"] == "Testing"),
                       key=lambda x: x["source"])
train_pool = sorted((x for x in cleaned if x["original_partition"] == "Training"),
                    key=lambda x: x["sha256"])
if (len(cleaned), len(train_pool), len(original_test), conflicts) != (2910, 2447, 463, 6):
    raise ValueError("Dataset differs from audited Stage 1 source; check Kaggle dataset version. "
                     f"Pre-visual counts: total={len(cleaned)}, pool={len(train_pool)}, "
                     f"test={len(original_test)}, conflicting images={conflicts}.")

training, validation = train_test_split(
    train_pool, test_size=0.2, random_state=42,
    stratify=[x["class"] for x in train_pool])
training = sorted((x for x in training if x["source"] not in visual_exclusions),
                  key=lambda x: x["source"])
validation = sorted((x for x in validation if x["source"] not in visual_exclusions),
                    key=lambda x: x["source"])
assert (len(training), len(validation), len(original_test)) == (1956, 487, 463)
assert len({x["sha256"] for x in training + validation + original_test}) == 2906

split_rows = []
for name, group in [("Training", training), ("Validation", validation), ("Testing", original_test)]:
    for item in group:
        split_rows.append({"split": name, "source_path": item["source"],
                           "class": item["class"], "sha256": item["sha256"]})
split_df = pd.DataFrame(split_rows).sort_values(["split", "source_path"])
split_df.to_csv(OUT / "REMNET_Split_Used.csv", index=False)
split_sha = hashlib.sha256(split_df.to_csv(index=False).encode()).hexdigest()
print("Split: Training",len(training),"Validation",len(validation),"Testing",len(original_test))
# Compact integrity check: the original Stage 1 source, members and
# 1,956/487/463 assignments must be IDENTICAL on every execution.
EXPECTED_SPLIT_SHA = "9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792"
if split_sha != EXPECTED_SPLIT_SHA:
    raise RuntimeError("The rebuilt split differs from frozen Stage 1 image membership.")
print("Stage 1 split integrity verified:", split_sha)

# ----------------------------
# Reproducibility, transforms, and datasets
# ----------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU to perform the controlled energy experiment.")
print("GPU:", torch.cuda.get_device_name(0), "| Visible GPUs:", torch.cuda.device_count())

train_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
test_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)])

class ImageList(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as im:
            image = self.transform(im.convert("RGB"))
        return image, CLASS_TO_ID[row["class"]]

train_ds = ImageList(training, train_tf)
val_ds = ImageList(validation, test_tf)
test_ds = ImageList(original_test, test_tf)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_model(pretrained):
    return timm.create_model("efficientnet_lite0", pretrained=pretrained,
                             num_classes=len(CLASS_NAMES)).to(device)

def evaluate_loss(model, loader, criterion):
    model.eval(); total = 0.0
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            total += criterion(logits, labels.to(device)).item() * len(labels)
    return total / len(loader.dataset)

def make_tracker(subfolder, name):
    subfolder.mkdir(parents=True, exist_ok=True)
    # Keep an incomplete previous run separate from the current session.
    existing = subfolder / "emissions.csv"
    if existing.exists():
        existing.rename(subfolder / f"emissions_previous_{time.time_ns()}.csv")
    return EmissionsTracker(project_name=name, output_dir=str(subfolder),
                            output_file="emissions.csv", save_to_file=True,
                            tracking_mode="machine", gpu_ids=[0],
                            measure_power_secs=MONITOR_SECONDS, pue=1.0,
                            log_level="error")

def measured_energy(subfolder):
    csv_file = subfolder / "emissions.csv"
    if not csv_file.exists():
        raise FileNotFoundError(f"CodeCarbon output missing: {csv_file}")
    row = pd.read_csv(csv_file).iloc[-1]
    energy = float(row["energy_consumed"])  # Final cumulative value; never sum rows.
    if not np.isfinite(energy) or energy < 0:
        raise ValueError(f"Invalid CodeCarbon energy: {energy}")
    return energy

# ----------------------------
# Brief Kaggle pilot: training/validation image loading, model forward,
# and an original CodeCarbon CSV. Pilot does not inspect the held-out test set.
# ----------------------------
if not (OUT / "EfficientNet_Lite0_pilot_ok.json").exists():
    set_seed(7)
    pilot_model = make_model(pretrained=True).eval()
    batch, _ = next(iter(DataLoader(val_ds, batch_size=2, shuffle=False)))
    pilot_log = OUT / "CodeCarbon_Logs" / "EfficientNet_Lite0_pilot"
    tracker = make_tracker(pilot_log, "EfficientNet_Lite0_pilot")
    tracker.start()
    try:
        with torch.no_grad():
            result = pilot_model(batch.to(device))
        torch.cuda.synchronize()
    finally:
        tracker.stop()
    if tuple(result.shape) != (2, 6) or not (pilot_log / "emissions.csv").exists():
        raise RuntimeError("Kaggle pilot failed: model output or CodeCarbon log missing")
    (OUT / "EfficientNet_Lite0_pilot_ok.json").write_text(json.dumps({"split_sha256": split_sha,
        "gpu": torch.cuda.get_device_name(0), "codecarbon_version":installed_version("codecarbon")}, indent=2))
    del pilot_model
    torch.cuda.empty_cache()
    print("Pilot passed: image loading, 6-class output, CodeCarbon CSV.")
else:
    pilot_info = json.loads((OUT / "EfficientNet_Lite0_pilot_ok.json").read_text())
    if pilot_info["split_sha256"] != split_sha:
        raise RuntimeError("Previous pilot used a different split. Remove old outputs or restore split.")
    print("Reusing pilot record for identical split.")

# ----------------------------
# Train EfficientNet-Lite0 once per seed using validation-loss selection.
# Reuse matching checkpoint + original training record on rerun.
# ----------------------------
def prepare_model(seed):
    name = "EfficientNet-Lite0"
    prefix = f"{name}_seed{seed}"
    ckpt_file = OUT / "Model_Checkpoints" / f"{prefix}.pt"
    record_file = OUT / "Model_Checkpoints" / f"{prefix}_training.json"
    log_dir = OUT / "CodeCarbon_Logs" / prefix / "training"
    if ckpt_file.exists() and record_file.exists() and (log_dir / "emissions.csv").exists():
        saved = json.loads(record_file.read_text())
        if saved.get("split_sha256") != split_sha or saved.get("seed") != seed or saved.get("model") != name:
            raise RuntimeError(f"Existing {name} seed {seed} checkpoint belongs to a different experiment")
        model = make_model(pretrained=False)
        obj = torch.load(ckpt_file, map_location="cpu", weights_only=True)
        if obj.get("split_sha256") != split_sha or obj.get("seed") != seed:
            raise RuntimeError("Checkpoint metadata mismatch")
        model.load_state_dict(obj["state_dict"])
        return model.eval(), saved

    model = make_model(pretrained=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    best_loss, best_epoch, epochs_run, stale = float("inf"), 0, 0, 0
    best_weights = None
    tracker = make_tracker(log_dir, prefix + "_training")
    torch.cuda.synchronize(); tracker.start(); start = time.perf_counter()
    try:
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward(); optimizer.step()
            val_loss = evaluate_loss(model, val_loader, criterion)
            epochs_run = epoch
            print(f"{prefix}: completed epoch {epoch}/{MAX_EPOCHS}; checkpoint selected by validation loss")
            if val_loss < best_loss - MIN_DELTA:
                best_loss, best_epoch, stale = val_loss, epoch, 0
                best_weights = copy.deepcopy(model.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break
        model.load_state_dict(best_weights)
        torch.cuda.synchronize()
    finally:
        seconds = time.perf_counter() - start
        tracker.stop()
    energy = measured_energy(log_dir)
    saved = {"model":name,"seed":seed,"split_sha256":split_sha,
             "epochs_run":epochs_run,"best_epoch":best_epoch,"best_val_loss":best_loss,
             "training_energy_kWh":energy,"training_co2_kg":energy*CARBON_FACTOR,
             "training_seconds":seconds,"monitoring_seconds":MONITOR_SECONDS,
             "codecarbon_csv":str(log_dir / "emissions.csv")}
    torch.save({"state_dict":model.state_dict(),"seed":seed,"split_sha256":split_sha,
                "model":name,"best_epoch":best_epoch},ckpt_file)
    record_file.write_text(json.dumps(saved, indent=2))
    return model.eval(), saved

# ----------------------------
# ----------------------------
# Held-out inference: one EfficientNet-Lite0 model.
# Training and inference energy are tracked in separate CodeCarbon sessions.
# ----------------------------
def evaluate_efficientnet(seed, model):
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model.eval()
    first_batch, _ = next(iter(loader))
    with torch.no_grad():
        for _ in range(2):
            model(first_batch.to(device))
    torch.cuda.synchronize()
    log_dir = OUT / "CodeCarbon_Logs" / f"EfficientNet-Lite0_seed{seed}" / "inference"
    tracker = make_tracker(log_dir, f"EfficientNet-Lite0_seed{seed}_inference")
    tracker.start(); torch.cuda.synchronize(); start = time.perf_counter()
    repeats = 0
    try:
        with torch.no_grad():
            while repeats < MIN_INFERENCE_REPEATS or time.perf_counter() - start < 31:
                for images, _ in loader:
                    model(images.to(device))
                repeats += 1
            torch.cuda.synchronize()
    finally:
        seconds = time.perf_counter() - start
        tracker.stop()
    energy = measured_energy(log_dir)
    return {"seed": seed,
            "inference_energy_kWh": energy,
            "inference_co2_kg": energy * CARBON_FACTOR,
            "inference_seconds": seconds,
            "inference_repetitions": repeats,
            "unique_test_images": len(test_ds), "batch_size": BATCH_SIZE,
            "monitor_interval_seconds": MONITOR_SECONDS,
            "monitored_image_evaluations": repeats * len(test_ds),
            "inference_energy_kWh_per_image": energy / (repeats * len(test_ds)),
            "inference_seconds_per_image": seconds / (repeats * len(test_ds))}

results = []
for seed in SEEDS:
    print("\n==== EfficientNet-Lite0 SEED", seed, "====", flush=True)
    set_seed(seed)
    model, training = prepare_model(seed)
    row = evaluate_efficientnet(seed, model)
    row["training_energy_kWh"] = training["training_energy_kWh"]
    row["training_co2_kg"] = training["training_co2_kg"]
    row["training_seconds"] = training["training_seconds"]
    row["epochs_run"] = training["epochs_run"]
    row["best_epoch"] = training["best_epoch"]
    row["total_energy_kWh"] = row["training_energy_kWh"] + row["inference_energy_kWh"]
    row["total_co2_kg"] = row["training_co2_kg"] + row["inference_co2_kg"]
    row["total_seconds"] = row["training_seconds"] + row["inference_seconds"]
    row["carbon_factor_kg_per_kWh"] = CARBON_FACTOR
    row["codecarbon_version"] = installed_version("codecarbon")
    row["gpu"] = torch.cuda.get_device_name(0)
    row["monitor_interval_seconds"] = MONITOR_SECONDS
    results.append(row)
    print(f"Seed {seed} | train: {row['training_energy_kWh']:.8f} kWh, "
          f"{row['training_co2_kg']:.8f} kg CO2e, {row['training_seconds']:.2f}s, "
          f"{row['epochs_run']} epochs | inference: "
          f"{row['inference_energy_kWh']:.8f} kWh, "
          f"{row['inference_co2_kg']:.8f} kg CO2e, "
          f"{row['inference_seconds']:.2f}s", flush=True)
    del model
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
for label, prefix in [("TRAINING", "training"), ("INFERENCE", "inference"),
                      ("TRAINING + ONE INFERENCE SESSION", "total")]:
    print("\n" + "=" * 71)
    print(f"FINAL 5-SEED SUMMARY — EfficientNet-Lite0 | {label}")
    print("=" * 71)
    print(f"{'Seed':>6} {'CO2e (kg)':>14} {'Energy (kWh)':>18} {'Time (min)':>15}")
    for row in results:
        print(f"{row['seed']:>6} {row[prefix+'_co2_kg']:>14.8f} "
              f"{row[prefix+'_energy_kWh']:>18.8f} "
              f"{row[prefix+'_seconds']/60:>15.5f}")
    print("-" * 71)
    print(f"Mean CO2e (kg):     {np.mean([r[prefix+'_co2_kg'] for r in results]):.8f}")
    print(f"Mean Energy (kWh):  {np.mean([r[prefix+'_energy_kWh'] for r in results]):.8f}")
    print(f"Mean Time (min):    {np.mean([r[prefix+'_seconds']/60 for r in results]):.5f}")
print("\nEpochs per seed:", {r['seed']: r['epochs_run'] for r in results})
print(f"Inference: {len(test_ds)} test images, batch {BATCH_SIZE}, "
      f"at least {MIN_INFERENCE_REPEATS} passes and 31 seconds; "
      f"CodeCarbon sampling {MONITOR_SECONDS}s, GPU 0 only.")
print(f"Carbon factor: {CARBON_FACTOR} kg CO2e/kWh (fixed assumption, not regional).")
print("Original CodeCarbon CSV logs and model checkpoints preserved under:", OUT)

# Preserve per-seed results for reproducibility while displaying all summaries in the console.
results_df.to_csv(OUT / "EfficientNet_Lite0_Energy_CO2.csv", index=False)

hardware = {"platform":"Kaggle", "python":platform.python_version(),
            "torch":torch.__version__,"torchvision": __import__("torchvision").__version__,
            "timm":timm.__version__,"codecarbon":installed_version("codecarbon"),
            "gpu":torch.cuda.get_device_name(0), "visible_gpu_count":torch.cuda.device_count(),
            "carbon_factor_kg_per_kWh":CARBON_FACTOR,
            "carbon_factor_note":"Fixed global reference; actual Kaggle datacenter carbon intensity unknown",
            "power_sampling_seconds":MONITOR_SECONDS,"batch_size":BATCH_SIZE,
            "split_sha256":split_sha,"unique_test_images":463,
            "model":"EfficientNet-Lite0",
             "split_protocol":"Exact Stage 1 membership reconstructed: SHA-256 sorting, stratified 80/20 split random_state=42, then four visual exclusions"}
(OUT / "EfficientNet_Lite0_Experiment_Configuration.json").write_text(json.dumps(hardware, indent=2))

report = Document()
report.add_heading("EfficientNet-Lite0 — Stage 2 Sustainability Summary", 0)
report.add_paragraph("Frozen dataset: 1,956 training, 487 validation, 463 held-out testing images. "
    "Five seeds: 7, 49, 112, 345, 476. Validation-loss checkpoint selection, "
    "32x32 RGB images, batch size 32, and Adam learning rate 1e-4.")
report.add_paragraph("The Stage 1 image assignments are reconstructed from the original "
    "Kaggle source and verified by split checksum. Training and held-out inference "
    "are measured in separate CodeCarbon sessions. Inference: 463 unique test images, "
    "batch size 32, at least 20 passes and at least 31 seconds of monitoring; "
    "GPU 0 only; CodeCarbon sampling interval 15 seconds. CO2e uses a fixed "
    "global factor of 0.475 kg/kWh, not a verified Kaggle location factor.")
table = report.add_table(rows=1, cols=8)
for cell, label in zip(table.rows[0].cells,
    ["Seed", "Epochs", "Train kWh", "Train CO2e kg", "Train s",
     "Infer kWh", "Infer CO2e kg", "Infer s"]):
    cell.text = label
for _, r in results_df.iterrows():
    cells = table.add_row().cells
    for cell, value in zip(cells, [str(int(r.seed)), str(int(r.epochs_run)),
                     f"{r.training_energy_kWh:.8f}", f"{r.training_co2_kg:.8f}",
                     f"{r.training_seconds:.2f}", f"{r.inference_energy_kWh:.8f}",
                     f"{r.inference_co2_kg:.8f}", f"{r.inference_seconds:.2f}"]):
        cell.text = value
report.add_paragraph("Original CodeCarbon CSV files, checkpoints and per-seed "
                     "training records are stored in the Stage 2 folder.")
report.save(OUT / "EfficientNet_Lite0_Stage2_Summary.docx")

print("\nResults displayed in the console; evidence files saved under:", OUT)


Source Training: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Training 
Source Testing: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Testing
Split: Training 1956 Validation 487 Testing 463
Stage 1 split integrity verified: 9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792
GPU: Tesla T4 | Visible GPUs: 1


model.safetensors: reconstructing file:   0%|          |  0.00B / 18.8MB            

model.safetensors: downloading bytes:           |  0.00B            

[codecarbon WARNING @ 07:38:42] Multiple instances of codecarbon are allowed to run at the same time.


Pilot passed: image loading, 6-class output, CodeCarbon CSV.

==== EfficientNet-Lite0 SEED 7 ====


EfficientNet-Lite0_seed7: completed epoch 1/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 2/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 3/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 4/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 5/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 6/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 7/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 8/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 9/10; checkpoint selected by validation loss
EfficientNet-Lite0_seed7: completed epoch 10/10; checkpoint selected by validation loss
Seed 7 | train: 0.00675305 kWh, 0.00320770 kg CO2e, 282.57s, 10 epochs | inference: 0.00128460 kWh, 0.00061018 kg CO2e, 60.49s

=

In [3]:
# ================================================================
# ADDITION ONLY: Test-set class probabilities (EfficientNet-Lite0)
# This cell runs AFTER all existing training and sustainability cells.
# It is NOT part of the timed/CodeCarbon-monitored inference session.
# Existing splits, training, checkpoints, CO2 estimates, and CSVs
# remain unchanged. All new CSVs go into Test_Probabilities/.
# ================================================================

probability_dir = OUT / "Test_Probabilities"
probability_dir.mkdir(parents=True, exist_ok=True)

# Use the same deterministic test transforms, order, and batch size.
probability_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
expected_labels = [CLASS_TO_ID[row["class"]] for row in original_test]
all_probability_frames = []

for probability_seed in SEEDS:
    checkpoint_file = (
        OUT / "Model_Checkpoints" /
        f"EfficientNet-Lite0_seed{probability_seed}.pt"
    )
    if not checkpoint_file.is_file():
        raise FileNotFoundError(
            f"Cannot export probabilities: missing {checkpoint_file}. "
            "Finish the unchanged Stage 2 training cell first."
        )

    # Load the exact frozen best-validation checkpoint; no retraining.
    checkpoint = torch.load(
        checkpoint_file, map_location="cpu", weights_only=True
    )
    if (checkpoint.get("seed") != probability_seed or
            checkpoint.get("model") != "EfficientNet-Lite0" or
            checkpoint.get("split_sha256") != split_sha):
        raise RuntimeError(
            f"Checkpoint identity or dataset split mismatch: {checkpoint_file}"
        )

    probability_model = make_model(pretrained=False)
    try:
        probability_model.load_state_dict(checkpoint["state_dict"])
        probability_model.eval()
        probability_batches = []
        observed_labels = []

        # Unmonitored prediction-only forward pass.
        with torch.inference_mode():
            for images, labels in probability_loader:
                logits = probability_model(images.to(device))
                probabilities = torch.softmax(logits, dim=1)
                probability_batches.append(probabilities.cpu().numpy())
                observed_labels.extend(labels.tolist())

        if observed_labels != expected_labels:
            raise RuntimeError("Test-image/label order changed; export aborted.")
        scores = np.concatenate(probability_batches, axis=0)
        if scores.shape != (len(original_test), len(CLASS_NAMES)):
            raise RuntimeError(f"Unexpected probability array shape: {scores.shape}")
        if not np.all(np.isfinite(scores)) or not np.allclose(
            scores.sum(axis=1), 1.0, atol=1e-5
        ):
            raise RuntimeError("Invalid probabilities: nonfinite or sum != 1.")

        predicted_ids = np.argmax(scores, axis=1)
        frame = pd.DataFrame({
            "seed": [probability_seed] * len(original_test),
            "source_path": [item["source"] for item in original_test],
            "true_class": [item["class"] for item in original_test],
            "true_class_id": observed_labels,
            "predicted_class": [CLASS_NAMES[int(i)] for i in predicted_ids],
            "predicted_class_id": predicted_ids,
            "prediction_confidence": scores.max(axis=1),
        })
        for class_index, class_name in enumerate(CLASS_NAMES):
            frame[f"prob_{class_name}"] = scores[:, class_index]

        probability_file = (
            probability_dir /
            f"EfficientNet_Lite0_seed{probability_seed}_test_probabilities.csv"
        )
        frame.to_csv(probability_file, index=False, float_format="%.10f")
        all_probability_frames.append(frame)
        print(f"Saved seed {probability_seed}: {len(frame)} test predictions -> {probability_file}")
    finally:
        del probability_model
        torch.cuda.empty_cache()

# One additional combined file: 463 held-out images × 5 seeds.
combined_file = probability_dir / "EfficientNet_Lite0_all_seeds_test_probabilities.csv"
pd.concat(all_probability_frames, ignore_index=True).to_csv(
    combined_file, index=False, float_format="%.10f"
)
print("Probability export complete; original sustainability measurements unchanged.")
print("Combined file:", combined_file)


Saved seed 7: 463 test predictions -> /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_seed7_test_probabilities.csv
Saved seed 49: 463 test predictions -> /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_seed49_test_probabilities.csv
Saved seed 112: 463 test predictions -> /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_seed112_test_probabilities.csv
Saved seed 345: 463 test predictions -> /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_seed345_test_probabilities.csv
Saved seed 476: 463 test predictions -> /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_seed476_test_probabilities.csv
Probability export complete; original sustainability measurements unchanged.
Combined file: /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_all_seeds_test_probabilities.csv
